We start with filtering out the non-tubingensis species from our data

In [ ]:
bcftools view \ 
    -S ^"/home/puntm/my_scratch_dir/2026_black_aspergillus/data/no_atubingensis.list" \
    -O z \
    -o /home/puntm/my_scratch_dir/2026_refbased/filtered_species.vcf.gz \
    /home/puntm/my_scratch_dir/2026_refbased/all_scaffolds.filteredSNPs.vcf.gz

In [ ]:
plink \
  --allow-extra-chr \
  --make-bed \
  --set-missing-var-ids @:# \
  --out /home/puntm/my_scratch_dir/2026_refbased/plink/fspecies \
  --indep-pairwise 50 10 0.1 \
  --geno 0.1 \
  --maf 0.01 \
  --snps-only just-acgt \
  --vcf /home/puntm/my_scratch_dir/2026_refbased/filtered_species.vcf.gz \
  --vcf-half-call missing
#recode the : to tab
sed 's/:/\t/g' /home/puntm/my_scratch_dir/2026_refbased/plink/fspecies.prune.in > /home/puntm/my_scratch_dir/2026_refbased/plink/targets.txt

bcftools view -T /home/puntm/my_scratch_dir/2026_refbased/plink/targets.txt /home/puntm/my_scratch_dir/2026_refbased/filtered_species.vcf.gz --output /home/puntm/my_scratch_dir/2026_refbased/pruned.vcf.gz

/home/puntm/my_scratch_dir/local_software/vcf2phylip.py --input /home/puntm/my_scratch_dir/2026_refbased/pruned.vcf.gz -f 

Ok, now we have pruned the data for only the atubingensis species, now I will build a new tree to enable new PCA covariates to use as input in plink


In [ ]:
fasttree -gtr -nt < /home/puntm/my_scratch_dir/2026_refbased/pruned.min4.fasta > /home/puntm/my_scratch_dir/2026_black_aspergillus//data/tree_data/fspecies.tree

In [19]:
import pandas as pd
import numpy as np
import os

# Load the data
input_path = '../data/2026_meta_info.csv'
df = pd.read_csv(input_path)

# --- Configuration ---
strain_col = 'strain'
mic_cols = ['itc', 'pos', 'vrc'] 
# ---------------------

# 1. Initialize the master dataframe with PLINK ID columns
final_pheno = pd.DataFrame()
final_pheno['FID'] = df[strain_col]
final_pheno['IID'] = df[strain_col]


# 2. Loop through each drug and add the Log2 column
for drug in mic_cols:
    # Convert to numeric (handles 'NA' strings)
    numeric_mic = pd.to_numeric(df[drug], errors='coerce')
    
    # Calculate Log2 (using a mask to avoid log(0) errors if any MICs are 0)
    final_pheno[f'Log2_{drug}'] = np.log2(numeric_mic.replace(0, np.nan))

# 3. Handle Missing Data
# Replacing NaNs with -9 (Standard PLINK missing value indicator)
#final_pheno = final_pheno.fillna(int(0))
#final_pheno['FID'] = int(0)

# 4. Save to one single file
output_path = '../data/GWAS/all_mic_phenotypes.txt'

# Ensure the results directory exists
os.makedirs('../data/GWAS', exist_ok=True)

final_pheno.to_csv(output_path, sep='\t', index=False, quoting=3)

print(f"Successfully created {output_path} with traits: {', '.join(mic_cols)}")
print(final_pheno.head())

Successfully created ../data/GWAS/all_mic_phenotypes.txt with traits: itc, pos, vrc
       FID      IID  Log2_itc  Log2_pos  Log2_vrc
0  V273-64  V273-64       0.0      -2.0       0.0
1  M039-56  M039-56       1.0      -1.0       1.0
2  M039-58  M039-58       0.0       1.0      -2.0
3  V080-81  V080-81       0.0      -2.0       0.0
4  V072-57  V072-57       5.0       1.0      -2.0


The below script converts a nwk file to matrix format, I changed the input tree from the raxml one to a fasttree version that only uses tubingesis sensu stricto

In [20]:
from ete3 import Tree
import pandas as pd

t = Tree("../data/tree_data/fspecies.tree", format=1)
samples = [leaf.name for leaf in t.iter_leaves()]
distances = pd.DataFrame(index=samples, columns=samples)

for i in samples:
    for j in samples:
        try:
            node_i = t.search_nodes(name=i)[0]
            node_j = t.search_nodes(name=j)[0]

            distances.loc[i, j] = t.get_distance(node_i, node_j)
        except IndexError:
            print(f"Missing node: {i} or {j}")
            distances.loc[i, j] = float('nan')

distances.to_csv("../data/GWAS/species_distances.tsv", sep="\t")

Now we convert the distances to a PCA covariates format to be used in plink:

In [21]:
import pandas as pd
from sklearn.decomposition import PCA

dist = pd.read_csv("../data/GWAS/species_distances.tsv", sep="\t", index_col=0)
pca = PCA(n_components=10)
pcs = pca.fit_transform(dist)

# Convert to PLINK covariate format
# PLINK expects: FID, IID, PC1, PC2, ...
pca_df = pd.DataFrame(pcs, columns=[f'PC{i+1}' for i in range(10)])
pca_df.insert(0, 'IID', dist.index)
pca_df.insert(0, 'FID', dist.index)
# Save for PLINK
pca_df.to_csv("../data/GWAS/pca_covariates.txt", sep="\t", index=False)

In [ ]:
%%bash
VCF_FILE="/home/puntm/my_scratch_dir/2026_refbased/filtered_species.vcf.gz"

plink \
    --vcf "${VCF_FILE}" \
    --allow-extra-chr \
    --all-pheno \
    --linear \
    --allow-no-sex \
    --pheno "../data/GWAS/all_mic_phenotypes.txt" \
    --covar "../data/GWAS/pca_covariates.txt" \
    --out GWAS



bash: line 1: VCF_FILE: command not found
bash: line 4: plink: command not found


CalledProcessError: Command 'b'VCF_FILE ="$1"\n\nplink \\\n    --vcf "${VCF_FILE}" \\\n    --allow-extra-chr \\\n    --all-pheno \\\n    --linear \\\n    --no-sex \\\n    --assoc \\\n    --pheno "../results/all_mic_phenotypes.txt" \\\n    --covar "../results/pca_covariates.txt" \\\n    --out GWAS\n\n\n\n'' returned non-zero exit status 127.

Now we filter the only relevant info from thw GWAS:

# Keep the header and only rows that have 'ADD'
awk 'NR==1 || $5=="ADD" {print $0}' GWAS.Log2_itc.assoc.linear | gzip -9 > ../data/GWAS/itc.out.gz
awk 'NR==1 || $5=="ADD" {print $0}' GWAS.Log2_pos.assoc.linear | gzip -9 > ../data/GWAS/pos.out.gz
awk 'NR==1 || $5=="ADD" {print $0}' GWAS.Log2_vrc.assoc.linear | gzip -9 > ../data/GWAS/vrc.out.gz
